In [1]:
import os
import pandas as pd
from pathlib import Path
import gc

# Run garbage collection
gc.collect()

data_folder = Path(r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\data\train.csv")
data_path = os.path.join(os.getcwd(), data_folder)

print(data_path)

# # submission data format
# df = pd.read_parquet(data_path)
# df.head()

df = pd.read_csv(data_path)
df.head()

C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\data\train.csv


,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",qwen3.8-max
1,0041ac86ee93,p073,Data science communities matter because they a...,claude-sonnet-5
2,005a19083eae,p120,"In the final twilight of time, the last embers...",gemini-3.6-flash
3,009319596f10,p073,Data science communities matter because they t...,gpt-5.6-terra
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",glm-5.3-flash


In [2]:
# !pip install transformers 
# !pip install evaluate
# !pip install datasets 
# !pip install accelerate 
# %pip install sentencepiece protobuf blobfile
# %pip install torch torchvision torchaudio



In [3]:
import os
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, recall_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding

# 1. Encode Labels (0 to 17)
le = LabelEncoder()
df["label"] = le.fit_transform(df["model"])
num_classes = len(le.classes_)

# 2. Stratified Train-Val Split (har model ka equal distribution maintain karne ke liye)
df_train, df_val = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df["label"]
)

print(f"Train Shape: {df_train.shape}, Val Shape: {df_val.shape}")
print(f"Total LLM Classes: {num_classes}")

# 3. Tokenizer Setup
MODEL_NAME = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


c:\Users\ps302\anaconda3\envs\genai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train Shape: (2159, 5), Val Shape: (540, 5)
Total LLM Classes: 18


In [4]:
# 5. Dataset Class
class LLMDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding=False,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = LLMDataset(df_train["text"].values, df_train["label"].values, tokenizer)
val_dataset = LLMDataset(df_val["text"].values, df_val["label"].values, tokenizer)

# 6. Scikit-learn based compute_metrics (Zero DLL dependencies!)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    rec = recall_score(labels, preds, average="macro")
    return {"accuracy": acc, "recall": rec}


In [7]:
# 1. Model Load
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=num_classes
)

# 2. Check Device (CUDA GPU or CPU)
use_cuda = torch.cuda.is_available()
print(f"CUDA Available: {use_cuda}")

# 3. Training Arguments (Tuned for DeBERTa)
training_args = TrainingArguments(
    output_dir="./deberta_llm_output",
    eval_strategy="epoch",            # Har epoch ke end par validation test karega
    save_strategy="epoch",            # Har epoch par checkpoint save karega
    learning_rate=2e-5,               # Transformers ke liye optimal LR
    per_device_train_batch_size=8,    # Agar OOM (Out of Memory) aaye toh 4 kar dena
    per_device_eval_batch_size=16,
    num_train_epochs=4,               # 4 epochs
    weight_decay=0.01,                # Regularization (overfitting rokne ke liye)
    # warmup_ratio=0.1,                 # Gradual LR warmup
    fp16=use_cuda,                    # GPU par fast mixed precision training
    load_best_model_at_end=True,      # Automatically best epoch model load karega
    metric_for_best_model="accuracy", # Best model selection metric
    greater_is_better=True,
    logging_steps=25,
    report_to="none"                  # Third-party trackers off
)

# 4. Trainer Initialize
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

# 5. Start Training!
trainer.train()


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 25492.73it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING  

CUDA Available: False


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [ ]:
# 1. Direct Evaluation
eval_results = trainer.evaluate()

print("\n" + "="*45)
print("--- FINAL DEBERTA VALIDATION SCORES ---")
print(f"Validation Loss:     {eval_results['eval_loss']:.4f}")
print(f"Validation Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"Validation Recall:   {eval_results['eval_recall']:.4f}")
print("="*45)


Accuracy Score: 0.0204
Recall Score: 0.0208


In [ ]:
from sklearn.metrics import classification_report

# Validation data par full predictions nikalein
val_preds_output = trainer.predict(val_dataset)
y_pred = np.argmax(val_preds_output.predictions, axis=1)
y_true = val_preds_output.label_ids

print("\n--- Model-wise Classification Report ---")
print(classification_report(y_true, y_pred, target_names=le.classes_))


In [ ]:
# !mkdir -p wheels
# !pip download psycopg -d wheels

A subdirectory or file -p already exists.
Error occurred while processing: -p.


  Using cached psycopg-3.3.6-py3-none-any.whl.metadata (4.4 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached tzdata-2026.5-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached psycopg-3.3.6-py3-none-any.whl (215 kB)
Using cached typing_extensions-4.16.0-py3-none-any.whl (45 kB)
Using cached tzdata-2026.5-py2.py3-none-any.whl (347 kB)
Saved .\wheels\psycopg-3.3.6-py3-none-any.whl
Saved .\wheels\typing_extensions-4.16.0-py3-none-any.whl
Saved .\wheels\tzdata-2026.5-py2.py3-none-any.whl
Successfully downloaded psycopg typing-extensions tzdata
